# Hạ random_forest_regressor 

## **Các bước thực hiện:** 

#### **1. Khai báo thư viện, đọc data từ file csv**

In [1]:
# Đọc file
import pandas as pd

filepath = '../data/processed/'
df_test = pd.read_csv(filepath + 'test_data_final.csv')
df_train = pd.read_csv(filepath + 'train_data_final.csv')

In [2]:
df_train.head()

,price,original_price,discount_rate,quantity_sold,rating_average,review_count,is_return_policy,is_freeship_xtra,is_authentic,image_count,...,category_root_name_Điện Thoại - Máy Tính Bảng,category_root_name_Điện Tử - Điện Lạnh,category_root_name_Đồ chơi - Mẹ & Bé,China,Japan,South Korea,Thailand,USA,Vietnam,Others
0,0.445108,0.438575,0.000000,3.828641,0.74,0.331717,1.0,1.0,1.0,0.411765,...,0,0,0,1,1,0,0,0,0,0.00
1,0.313109,0.343930,0.500000,3.828641,1.00,0.367314,1.0,0.0,0.0,0.000000,...,0,0,0,0,0,0,0,0,0,0.25
2,0.156620,0.148556,0.000000,0.000000,0.00,0.000000,0.0,1.0,1.0,0.176471,...,0,0,0,0,0,0,0,0,1,0.00
3,0.263237,0.330742,0.846154,3.218876,0.86,0.310416,1.0,1.0,1.0,0.294118,...,0,0,0,0,0,0,0,0,0,0.25
4,0.396891,0.390102,0.000000,1.945910,0.80,0.175253,1.0,1.0,1.0,0.235294,...,0,0,0,1,0,0,1,0,0,0.00


In [3]:
df_test.head()

,price,original_price,discount_rate,quantity_sold,rating_average,review_count,is_return_policy,is_freeship_xtra,is_authentic,image_count,...,category_root_name_Điện Thoại - Máy Tính Bảng,category_root_name_Điện Tử - Điện Lạnh,category_root_name_Đồ chơi - Mẹ & Bé,China,Japan,South Korea,Thailand,USA,Vietnam,Others
0,0.069133,0.060604,0.000000,3.218876,1.0,0.175253,1.0,1.0,1.0,0.117647,...,0,0,0,0,0,0,0,0,1,0.00
1,0.470614,0.464217,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,0.411765,...,0,0,0,0,0,0,0,0,1,0.00
2,0.000000,0.000000,0.000000,1.791759,0.0,0.000000,1.0,1.0,1.0,0.000000,...,0,0,0,0,0,0,0,0,1,0.00
3,0.443963,0.444183,0.096154,3.465736,0.9,0.382518,1.0,1.0,1.0,0.588235,...,0,0,0,0,0,0,0,0,0,0.25
4,0.659291,0.653896,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,0.352941,...,0,0,0,0,0,0,0,1,0,0.00


In [4]:
# Tách X, y cho tập train
X_train = df_train.drop(columns=['quantity_sold'])
y_train = df_train['quantity_sold']

# Tách X, y cho tập train
X_test = df_test.drop(columns=['quantity_sold'])
y_test = df_test['quantity_sold']

#### **2. Huấn luyện mô hình Decision Tree Regressor**

In [5]:
# Huấn luyện mô hình cây (Decision Tree Regressor)
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(
    random_state=42
)

dt_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

#### **3. Tinh chỉnh mô hình (Finetune)**

**3.1. Khai báo grid siêu tham số và chạy GridSearchCV**

In [6]:
from sklearn.model_selection import GridSearchCV

# Tham số mô hình 
param_grid = {
    'max_depth': [5, 10, 20, None], # Độ sâu tối đa
    'min_samples_split': [2, 5, 10], # Số mẫu tối thiểu để tách một node
    'min_samples_leaf': [1, 2, 5] # Số mẫu tối thiểu ở node lá
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42), # Mô hình cần tối ưu
    param_grid=param_grid,
    cv=5, # Cross validation
    scoring='neg_root_mean_squared_error', # Tiêu chí đánh giá
    n_jobs=-1 # Chế độ
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [5, 10, 20, None],
                         'min_samples_leaf': [1, 2, 5],
                         'min_samples_split': [2, 5, 10]},
             scoring='neg_root_mean_squared_error')

**3.2. Chọn mô hình tốt nhất**

In [7]:
# Lấy mô hình tốt nhất
best_model = grid_search.best_estimator_
print("Tham số tốt nhất:", grid_search.best_params_)

Tham số tốt nhất: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10}


#### **4. Dự đoán kết quả và lưu vào file csv**

**4.1. Dự đoán kết quả**

In [8]:
y_test_pred = best_model.predict(X_test)

**4.2. Lưu file**

In [9]:
# Tạo dataframe kết quả
result_df = pd.DataFrame({
    'quantity_sold_ground_truth': y_test.values,
    'quantity_sold_predicted': y_test_pred
})

# Lưu file vào folder modeling
result_df.to_csv('../modeling/quantity_sold_predictions.csv', index=False)
